# Strategy 1 (E1) — Results

**Pipeline:** spaCy lemmatisation + discriminative TF-IDF dictionary + Aho-Corasick matching + adapted NegEx negation.

This notebook reproduces the results reported for the rule-based asthma detector in the thesis. All logic lives in the `asthma_e1` package; the notebook only orchestrates and visualises.

**Sections**
- 0. Setup
- 1. Corpus loading and description *(5.1)*
- 2. Discriminative dictionary analysis *(5.1)*
- 3. Cross-validation metrics *(5.2.3)*
- 4. NegEx behaviour *(5.2)*
- 5. Error analysis — FN / FP *(5.2.4)*
- 6. Hyperparameter sensitivity *(5.2)*
- 7. Computational cost *(5.2)*

## 0. Setup

In [ ]:
import sys, time
from pathlib import Path
from collections import defaultdict, Counter

sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold

from asthma_e1 import (CONFIG, PATHS, load_dataset_records, load_spacy_model,
                       build_spacy_docs, normalize_doc, build_corpus_map,
                       build_patient_level_texts, select_tfidf_terms,
                       build_norm_to_raw, predict_documents,
                       aggregate_patient_predictions, compute_patient_metrics)

sns.set_theme(style='whitegrid', context='notebook')
PATHS.figures_dir.mkdir(parents=True, exist_ok=True)
PATHS.tables_dir.mkdir(parents=True, exist_ok=True)
COLORS = {'asthma': '#2C5F8A', 'no_asthma': '#A8C5DA'}
print('Config:', CONFIG)

## 1. Corpus loading and description  *(5.1)*

Load and clean the corpus, lemmatise with spaCy, and build the lemma to raw-form map. This step runs once; later sections reuse these objects.

In [ ]:
records = load_dataset_records(PATHS.dataset_dir, CONFIG.exclude_patients)
print(f'{len(records)} documents after cleaning and patient exclusion')

nlp = load_spacy_model()
t0 = time.time()
docs_spacy = build_spacy_docs(records, nlp)
norm_docs = [normalize_doc(d) for d in docs_spacy]
corpus_map = build_corpus_map(docs_spacy)
print(f'Lemmatisation + corpus map: {time.time()-t0:.1f}s ({len(corpus_map)} lemmas)')

In [ ]:
# Table 1: corpus overview
df_records = pd.DataFrame([{'patient_id': r.patient_id, 'label': r.label,
                            'file_name': r.file_name,
                            'n_tokens': len(r.text.split())} for r in records])

n_pat = df_records['patient_id'].nunique()
n_pos = df_records[df_records['label']==1]['patient_id'].nunique()
n_neg = n_pat - n_pos
docs_per_pat = df_records.groupby('patient_id').size()

summary = pd.DataFrame({
    'Metric': ['Total patients', 'Active asthma (positives)', 'Non-asthma (negatives)',
               'Prevalence (%)', 'Total documents', 'Documents per patient (median)',
               'Tokens per document (median)', 'Tokens per patient (median)'],
    'Value': [n_pat, n_pos, n_neg, round(100*n_pos/n_pat, 1), len(df_records),
              int(docs_per_pat.median()), int(df_records['n_tokens'].median()),
              int(df_records.groupby('patient_id')['n_tokens'].sum().median())],
})
summary.to_csv(PATHS.tables_dir / 'corpus_summary.csv', index=False)
summary

In [ ]:
# Figure: document length and documents per patient
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for lbl, name, color in [(1, 'Active asthma', COLORS['asthma']), (0, 'No asthma', COLORS['no_asthma'])]:
    axes[0].hist(df_records[df_records['label']==lbl]['n_tokens'], bins=40,
                 alpha=0.85, label=name, color=color, edgecolor='white', linewidth=0.4)
axes[0].set_xlabel('Tokens per document'); axes[0].set_ylabel('Frequency')
axes[0].set_title('(a) Document length'); axes[0].legend()

dpp = df_records.groupby(['patient_id', 'label']).size().reset_index(name='n')
for lbl, name, color in [(1, 'Active asthma', COLORS['asthma']), (0, 'No asthma', COLORS['no_asthma'])]:
    axes[1].hist(dpp[dpp['label']==lbl]['n'], bins=range(1, int(dpp['n'].max())+2),
                 alpha=0.85, label=name, color=color, edgecolor='white', linewidth=0.4)
axes[1].set_xlabel('Documents per patient'); axes[1].set_ylabel('No. patients')
axes[1].set_title('(b) Documents per patient'); axes[1].legend()
for ax in axes:
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(PATHS.figures_dir / 'fig_corpus_overview.png', dpi=200)
plt.show()

## 2. Discriminative dictionary analysis  *(5.1)*

The dictionary is selected per training fold (to avoid leakage). Here we inspect which terms are chosen and how stable they are across folds.

In [ ]:
pat_texts, pat_labels, pat_ids = build_patient_level_texts(records, norm_docs)
skf = StratifiedKFold(n_splits=CONFIG.cv_n_splits, shuffle=True, random_state=CONFIG.random_state)
splits = list(skf.split(pat_ids, pat_labels))

fold_dicts = {}
for fold, (train_idx, _) in enumerate(splits, 1):
    tr_texts = [pat_texts[i] for i in train_idx]
    tr_labels = [pat_labels[i] for i in train_idx]
    fold_dicts[fold] = select_tfidf_terms(tr_texts, tr_labels, top_n=CONFIG.top_n_terms)
    print(f'  Fold {fold}: {fold_dicts[fold]}')

term_counts = Counter(t for terms in fold_dicts.values() for t in terms)
df_stab = (pd.DataFrame(term_counts.items(), columns=['term', 'n_folds'])
           .sort_values('n_folds', ascending=False).reset_index(drop=True))
df_stab['stability_pct'] = (df_stab['n_folds'] / CONFIG.cv_n_splits * 100).round(1)
df_stab.to_csv(PATHS.tables_dir / 'dictionary_stability.csv', index=False)
df_stab

In [ ]:
# Figure: dictionary stability across folds
fig, ax = plt.subplots(figsize=(7, max(3, 0.4*len(df_stab))))
df_plot = df_stab.sort_values('n_folds')
bar_colors = [COLORS['asthma'] if n == CONFIG.cv_n_splits else COLORS['no_asthma']
              for n in df_plot['n_folds']]
ax.barh(df_plot['term'], df_plot['n_folds'], color=bar_colors, edgecolor='white', linewidth=0.4)
ax.axvline(CONFIG.cv_n_splits, ls='--', color='grey', alpha=0.5)
ax.set_xlabel(f'Number of folds (max = {CONFIG.cv_n_splits})')
ax.set_title('Dictionary stability across folds')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(PATHS.figures_dir / 'fig_dictionary.png', dpi=200)
plt.show()

## 3. Cross-validation metrics  *(5.2.3)*

Patient-level stratified 5-fold CV. We store document- and patient-level predictions for the later analyses.

In [ ]:
pat_to_idx = defaultdict(list)
for i, rec in enumerate(records):
    pat_to_idx[rec.patient_id].append(i)
pat_text_by_id = dict(zip(pat_ids, pat_texts))
pat_label_by_id = dict(zip(pat_ids, pat_labels))

fold_metrics, all_doc_preds, all_pat_preds, fold_times = [], [], [], []
for fold, (train_idx, test_idx) in enumerate(splits, 1):
    t_fold = time.time()
    tr_texts = [pat_text_by_id[pat_ids[i]] for i in train_idx]
    tr_labels = [pat_label_by_id[pat_ids[i]] for i in train_idx]
    test_ids = [pat_ids[i] for i in test_idx]

    test_doc_idx = sorted(i for pid in test_ids for i in pat_to_idx[pid])
    test_records = [records[i] for i in test_doc_idx]
    test_docs = [docs_spacy[i] for i in test_doc_idx]
    test_norm = [norm_docs[i] for i in test_doc_idx]

    dictionary = select_tfidf_terms(tr_texts, tr_labels, top_n=CONFIG.top_n_terms)
    norm_to_raw = build_norm_to_raw(dictionary, corpus_map)
    doc_preds = predict_documents(test_records, test_docs, test_norm, dictionary, norm_to_raw)
    pat_preds = aggregate_patient_predictions(doc_preds)

    m = compute_patient_metrics(pat_preds); m['fold'] = fold
    fold_metrics.append(m)
    all_doc_preds += [{'fold': fold, **d.__dict__} for d in doc_preds]
    all_pat_preds += [{'fold': fold, **p.__dict__} for p in pat_preds]
    fold_times.append(time.time() - t_fold)
    print(f"  Fold {fold}: P={m['precision']:.3f} R={m['recall']:.3f} F1={m['f1']:.3f} ({fold_times[-1]:.1f}s)")

df_fold = pd.DataFrame(fold_metrics)
df_doc = pd.DataFrame(all_doc_preds)
df_pat = pd.DataFrame(all_pat_preds)

In [ ]:
# Per-fold summary (mean +/- std)
metric_cols = ['precision', 'recall', 'f1', 'specificity', 'accuracy']
summ = df_fold[metric_cols].agg(['mean', 'std']).T
summ.columns = ['Mean', 'Std']
summ['Result'] = summ.apply(lambda r: f"{r['Mean']:.3f} ± {r['Std']:.3f}", axis=1)
summ.to_csv(PATHS.tables_dir / 'metrics_cv_summary.csv')
print(df_fold[['fold'] + metric_cols].round(3).to_string(index=False))
print('\n5-fold CV summary:')
print(summ[['Result']].to_string())

In [ ]:
# Pooled out-of-fold confusion matrix
from asthma_e1.detection import PatientPrediction
oof = [PatientPrediction(p['patient_id'], p['label'], p['prediction'], 0, 0) for p in all_pat_preds]
om = compute_patient_metrics(oof)
cm = np.array([[int(om['tn']), int(om['fp'])], [int(om['fn']), int(om['tp'])]])

fig, ax = plt.subplots(figsize=(5.2, 4.2))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Pred: No asthma', 'Pred: Asthma'],
            yticklabels=['True: No asthma', 'True: Asthma'], ax=ax, annot_kws={'size': 14})
ax.set_title('Confusion matrix (OOF)')
plt.tight_layout()
plt.savefig(PATHS.figures_dir / 'fig_confusion_matrix.png', dpi=200)
plt.show()
print({k: int(om[k]) for k in ['tp','fp','fn','tn']})

## 4. NegEx behaviour  *(5.2)*

How many dictionary-term mentions are detected, and how many are discarded as negated.

In [ ]:
term_pos, term_neg = Counter(), Counter()
for d in all_doc_preds:
    for t in d['positives']: term_pos[t] += 1
    for t in d['negated']:   term_neg[t] += 1

n_pos_m, n_neg_m = sum(term_pos.values()), sum(term_neg.values())
rate = n_neg_m / (n_pos_m + n_neg_m) if (n_pos_m + n_neg_m) else 0
print(f'Affirmed mentions: {n_pos_m}')
print(f'Negated mentions:  {n_neg_m}')
print(f'Negation rate:     {100*rate:.1f}%')

all_terms = set(term_pos) | set(term_neg)
df_negex = pd.DataFrame([{'term': t, 'n_affirmed': term_pos[t], 'n_negated': term_neg[t],
                          'total': term_pos[t]+term_neg[t]} for t in all_terms]
                        ).sort_values('total', ascending=False)
df_negex.to_csv(PATHS.tables_dir / 'negex_per_term.csv', index=False)
df_negex

In [ ]:
# Figure: affirmed vs negated mentions per term
fig, ax = plt.subplots(figsize=(8, max(3, 0.45*len(df_negex))))
dfp = df_negex.sort_values('total')
ax.barh(dfp['term'], dfp['n_affirmed'], label='Affirmed', color=COLORS['asthma'], edgecolor='white', linewidth=0.4)
ax.barh(dfp['term'], dfp['n_negated'], left=dfp['n_affirmed'], label='Negated', color=COLORS['no_asthma'], edgecolor='white', linewidth=0.4)
ax.set_xlabel('Number of mentions'); ax.set_title('Detected mentions per term: affirmed vs negated')
ax.legend(loc='lower right')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(PATHS.figures_dir / 'fig_negex_per_term.png', dpi=200)
plt.show()

## 5. Error analysis — FN / FP  *(5.2.4)*

Characterise false negatives (active asthma missed) and false positives. For false negatives we categorise the cause: no terminology detected (lexicon ceiling) vs. terminology only negated (possible NegEx false negative).

In [ ]:
df_pat['outcome'] = df_pat.apply(
    lambda r: 'TP' if r['label']==1 and r['prediction']==1
         else 'FN' if r['label']==1 and r['prediction']==0
         else 'FP' if r['label']==0 and r['prediction']==1 else 'TN', axis=1)
print(df_pat['outcome'].value_counts().to_string())

df_doc['n_pos'] = df_doc['positives'].apply(lambda x: len(x) if isinstance(x, list) else 0)
df_doc['n_neg'] = df_doc['negated'].apply(lambda x: len(x) if isinstance(x, list) else 0)
pat_terms = (df_doc.groupby('patient_id')
             .agg(total_pos_mentions=('n_pos','sum'), total_neg_mentions=('n_neg','sum'),
                  negated_terms=('negated', lambda L: sorted(set().union(*[set(x) for x in L if isinstance(x, list)]) if len(L) else set())))
             .reset_index())
df_err = df_pat.merge(pat_terms, on='patient_id', how='left')

In [ ]:
# False negatives + categorisation
df_fn = df_err[df_err['outcome'] == 'FN'].copy()
def categorise(row):
    if row['total_pos_mentions'] == 0 and row['total_neg_mentions'] == 0:
        return 'A: no terminology detected'
    if row['total_pos_mentions'] == 0 and row['total_neg_mentions'] > 0:
        return 'B: only negated mentions (possible NegEx FN)'
    return 'C: other'
df_fn['error_category'] = df_fn.apply(categorise, axis=1)
df_fn[['patient_id','fold','total_pos_mentions','total_neg_mentions','negated_terms','error_category']].to_csv(PATHS.tables_dir / 'errors_FN.csv', index=False)
print(f'False negatives: {len(df_fn)}')
print(df_fn['error_category'].value_counts().to_string())

df_fp = df_err[df_err['outcome'] == 'FP'].copy()
df_fp[['patient_id','fold','total_pos_mentions','total_neg_mentions']].to_csv(PATHS.tables_dir / 'errors_FP.csv', index=False)
print(f'\nFalse positives: {len(df_fp)}')
df_fn[['patient_id','total_pos_mentions','total_neg_mentions','error_category']]

## 6. Hyperparameter sensitivity  *(5.2)*

Sensitivity of the detector to the dictionary size (`top_n`), the minimum TF-IDF ratio (`min_ratio`) and the NegEx windows.

In [ ]:
def evaluate_cv(top_n=CONFIG.top_n_terms, min_ratio=CONFIG.min_ratio,
                min_support=CONFIG.min_support, neg_window=CONFIG.neg_window,
                neg_window_strong=CONFIG.neg_window_strong):
    from asthma_e1.negation import is_negated
    from asthma_e1.detection import build_automaton, detect
    rows = []
    for fold, (train_idx, test_idx) in enumerate(splits, 1):
        tr_texts = [pat_text_by_id[pat_ids[i]] for i in train_idx]
        tr_labels = [pat_label_by_id[pat_ids[i]] for i in train_idx]
        test_ids = [pat_ids[i] for i in test_idx]
        test_doc_idx = sorted(i for pid in test_ids for i in pat_to_idx[pid])
        test_records = [records[i] for i in test_doc_idx]
        test_docs = [docs_spacy[i] for i in test_doc_idx]
        test_norm = [norm_docs[i] for i in test_doc_idx]

        dictionary = select_tfidf_terms(tr_texts, tr_labels, top_n=top_n,
                                        min_support=min_support, min_ratio=min_ratio)
        if not dictionary:
            rows.append({'precision':0,'recall':0,'f1':0,'n_dict':0}); continue
        norm_to_raw = build_norm_to_raw(dictionary, corpus_map)
        automaton = build_automaton(dictionary)
        pats = defaultdict(lambda: {'label': None, 'pred': 0})
        for rec, dsp, tn in zip(test_records, test_docs, test_norm):
            pats[rec.patient_id]['label'] = rec.label
            for term in detect(tn, automaton):
                raw = norm_to_raw.get(term, [term])
                if not is_negated(rec.text, dsp, raw, window=neg_window, window_strong=neg_window_strong):
                    pats[rec.patient_id]['pred'] = 1; break
        tp = sum(1 for v in pats.values() if v['label']==1 and v['pred']==1)
        fn = sum(1 for v in pats.values() if v['label']==1 and v['pred']==0)
        fp = sum(1 for v in pats.values() if v['label']==0 and v['pred']==1)
        prec = tp/(tp+fp) if (tp+fp) else 0
        rec_ = tp/(tp+fn) if (tp+fn) else 0
        f1 = 2*prec*rec_/(prec+rec_) if (prec+rec_) else 0
        rows.append({'precision':prec,'recall':rec_,'f1':f1,'n_dict':len(dictionary)})
    return pd.DataFrame(rows).mean().to_dict()

df_topn = pd.DataFrame([{**evaluate_cv(top_n=n), 'top_n': n} for n in [1,2,3,5,7,10,15,20]])
df_ratio = pd.DataFrame([{**evaluate_cv(min_ratio=r), 'min_ratio': r} for r in [1,2,5,10,20,30,50,100]])
df_neg = pd.DataFrame([{**evaluate_cv(neg_window=w, neg_window_strong=ws), 'window': w, 'window_strong': ws}
                       for w, ws in [(3,5),(5,10),(7,14),(10,20),(15,30)]])
df_topn.to_csv(PATHS.tables_dir / 'sensitivity_top_n.csv', index=False)
df_ratio.to_csv(PATHS.tables_dir / 'sensitivity_min_ratio.csv', index=False)
df_neg.to_csv(PATHS.tables_dir / 'sensitivity_negex_window.csv', index=False)
print('top_n sweep:'); print(df_topn.round(3).to_string(index=False))

In [ ]:
# Figure: hyperparameter sensitivity
CM = {'precision': '#0072B2', 'recall': '#E69F00', 'f1': '#009E73'}
MK = {'precision': 'o', 'recall': 's', 'f1': '^'}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for met in ['precision','recall','f1']:
    axes[0].plot(df_topn['top_n'], df_topn[met], marker=MK[met], label=met.title(), color=CM[met], lw=2, ms=7)
axes[0].set_xlabel('top_n (dictionary terms)'); axes[0].set_ylabel('Value (CV mean)')
axes[0].set_title('(a) Sensitivity to top_n'); axes[0].set_ylim(0, 1.05)
for met in ['precision','recall','f1']:
    axes[1].plot(df_ratio['min_ratio'], df_ratio[met], marker=MK[met], label=met.title(), color=CM[met], lw=2, ms=7)
axes[1].set_xscale('log'); axes[1].set_xlabel('min_ratio TF-IDF (log)')
axes[1].set_title('(b) Sensitivity to min_ratio'); axes[1].set_ylim(0, 1.05)
x = [f"({r['window']},{r['window_strong']})" for _, r in df_neg.iterrows()]
for met in ['precision','recall','f1']:
    axes[2].plot(x, df_neg[met], marker=MK[met], label=met.title(), color=CM[met], lw=2, ms=7)
axes[2].set_xlabel('(window, window_strong)')
axes[2].set_title('(c) Sensitivity to NegEx windows'); axes[2].set_ylim(0, 1.05)
for ax in axes:
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.grid(True, axis='y', ls='--', alpha=0.35)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.03))
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.savefig(PATHS.figures_dir / 'fig_sensitivity.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. Computational cost  *(5.2)*

In [ ]:
print('Fold times (s):')
for i, t in enumerate(fold_times, 1):
    print(f'  Fold {i}: {t:.2f}s')
print(f'  Mean: {np.mean(fold_times):.2f}s ± {np.std(fold_times):.2f}s')
pd.DataFrame({'fold': range(1, len(fold_times)+1), 'time_sec': fold_times}).to_csv(
    PATHS.tables_dir / 'timing_per_fold.csv', index=False)

---
### Output summary

Tables written to `results/tables/`: `corpus_summary.csv`, `dictionary_stability.csv`, `metrics_cv_summary.csv`, `negex_per_term.csv`, `errors_FN.csv`, `errors_FP.csv`, `sensitivity_top_n.csv`, `sensitivity_min_ratio.csv`, `sensitivity_negex_window.csv`, `timing_per_fold.csv`.

Figures written to `results/figures/`: `fig_corpus_overview.png`, `fig_dictionary.png`, `fig_confusion_matrix.png`, `fig_negex_per_term.png`, `fig_sensitivity.png`.